In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

sys.path.append("src")
from entropy_pruning import (
    UNILoRAClassifier,
    build_attention_cache,
    build_loaders,
    set_seed,
    train_forecaster,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
CFG = dict(
    data_dir="/data/NCT-CRC-HE",
    img_size=224,
    batch_size=64,
    num_workers=4,
    seed=42,
    layer_target=23,
    layers_source=[2],
    hidden=256,
    n_heads=4,
    n_layers=2,
    dropout=0.2,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
)

set_seed(CFG["seed"])
dataset_name = Path(CFG["data_dir"]).name
classifier_ckpt = Path(f"checkpoints/{dataset_name}/uni_finetuned/best_model.pt")
cache_path = Path(f"/data/data_cache/{dataset_name}_forecaster_dataset.h5")
forecaster_dir = Path(f"checkpoints/{dataset_name}/forecaster")
forecaster_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
loaders = build_loaders(
    data_dir=CFG["data_dir"],
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
    drop_last_train=False,
)
print("Classi:", loaders.class_names)
print("Train batches:", len(loaders.train_loader))


In [ ]:
model = UNILoRAClassifier(loaders.n_classes).to(device)
model.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

if not cache_path.exists():
    build_attention_cache(
        model=model,
        loaders={"train": loaders.train_loader, "val": loaders.val_loader, "test": loaders.test_loader},
        device=device,
        source_layers=CFG["layers_source"],
        target_layers=[CFG["layer_target"]],
        save_path=cache_path,
    )
print("Cache:", cache_path)


In [ ]:
all_results = []
for layer_source in CFG["layers_source"]:
    run_name = f"src{layer_source:02d}_tgt{CFG['layer_target']:02d}"
    result = train_forecaster(
        h5_cache_path=cache_path,
        layer_source=layer_source,
        layer_target=CFG["layer_target"],
        device=device,
        hidden=CFG["hidden"],
        n_heads=CFG["n_heads"],
        n_layers=CFG["n_layers"],
        dropout=CFG["dropout"],
        epochs=CFG["epochs"],
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        save_path=forecaster_dir / f"forecaster_{run_name}.pt",
    )
    all_results.append(result)
    print(result)


In [ ]:
layers = [r["layer_source"] for r in all_results]
rho_fore = [r["test_rho_forecaster"] for r in all_results]
rho_norm = [r["test_rho_token_norm"] for r in all_results]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(layers, rho_fore, marker="o", label="Forecaster")
ax.plot(layers, rho_norm, marker="s", linestyle="--", label="Token norm")
ax.set_xlabel("Layer sorgente")
ax.set_ylabel("Spearman rho")
ax.set_title(f"Target layer {CFG['layer_target']}")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
